# Preliminary Steps

Install ffmpeg

In [ ]:
%pip install pandas
%pip install matplotlib
%pip install ipywidgets
%pip install numpy

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display

input_folder = './csv'
processed_folder = './processed'
video_folder = './videos'
log_folder = './logs'

# Perform cleanup

import shutil
import os 

# Remove all files and subdirectories inside the video folder
if os.path.exists(video_folder):
    shutil.rmtree(video_folder)

    # Optionally, recreate the empty folder
    os.makedirs(video_folder)
    print(f"All files and subfolders in {video_folder} have been deleted.")
else:
    print(f"The folder {video_folder} does not exist.")

# Create checkbox for image generation
generate_image_checkbox = widgets.Checkbox(
    value=False,
    description='Generate Images'
)

# Create dropdown for animation type selection
animation_type_dropdown = widgets.Dropdown(
    options=['video', 'gif', 'both'],
    value='both',
    description='Animation Type'
)

# Create a text area for FPS input, with integer validation
fps_text_area = widgets.Text(
    value='1',  # Default value
    description='FPS:',
    disabled=False
)

# Function to validate FPS input (only allow integers)
def validate_fps(change):
    try:
        fps = int(change.new)
        if fps <= 0:
            fps_text_area.value = '1'  # Set default value if invalid
            print("Please enter a positive integer.")
    except ValueError:
        fps_text_area.value = '1'  # Reset to default if not an integer
        print("Invalid input. Please enter an integer.")

# Add the validation function to the text area
fps_text_area.observe(validate_fps, names='value')

# Create a button to trigger the animation generation
generate_button = widgets.Button(
    description='Generate Animations',
    button_style='success'
)

# Loop Over Input Files and Generate Animations (callback function)
def on_generate_button_click(b):
    # Get input folder files
    input_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    processed_files = [f for f in os.listdir(processed_folder) if f.endswith('_processed.csv')]

    # Loop over input files and generate animations
    for input_file in input_files:
        processed_file = f"{os.path.splitext(input_file)[0]}_processed.csv"
        if processed_file not in processed_files:
            continue
        generate_animation(input_file, processed_file)

# Set up the callback for the button
generate_button.on_click(on_generate_button_click)

# Display the widgets
display(generate_image_checkbox, animation_type_dropdown, fps_text_area, generate_button)

# Color Map

# Step 1: Generate a pool of 50 distinct colors using matplotlib
def generate_color_palette(num_colors=50):
    # Generate a colormap and sample num_colors evenly from it
    cmap = plt.get_cmap('tab20')  # You can use other colormaps like 'tab20', 'Set3', etc.
    colors = [cmap(i / num_colors) for i in range(num_colors)]
    return colors

# Step 2: Create color map for body parts
def create_body_part_color_map(body_parts, num_colors=50):
    # Generate a color pool
    color_pool = generate_color_palette(num_colors)
    
    # Ensure that the color pool is enough for the number of body parts
    if len(body_parts) > len(color_pool) // 2:
        raise ValueError("Not enough colors in the color pool for the number of body parts.")

    # Create a dictionary to store color assignments
    body_part_colors = {}
    
    # Map each body part to two colors, one for original and one for processed
    for i, body_part in enumerate(body_parts):
        # Use the index of the body part to determine the color pair (original and processed)
        original_color = color_pool[2 * i]  # Color for original data
        processed_color = color_pool[2 * i + 1]  # Color for processed data
        body_part_colors[body_part] = {
            'original': original_color,
            'processed': processed_color
        }
    
    return body_part_colors


def generate_animation(input_file, processed_file):
    # Extract widget values for use in code
    generate_images = generate_image_checkbox.value
    animation_type = animation_type_dropdown.value
    fps = int(fps_text_area.value)
    
    # Make sure output folder exists
    base_filename = os.path.splitext(input_file)[0]
    file_folder = os.path.join(video_folder, base_filename)
    os.makedirs(file_folder, exist_ok=True)
    image_folder = os.path.join(file_folder, 'images')
    os.makedirs(image_folder, exist_ok=True)
    
    # Read CSV files
    df_original = pd.read_csv(os.path.join(input_folder, input_file), header=[0, 1, 2])
    df_processed = pd.read_csv(os.path.join(processed_folder, processed_file), header=[0, 1, 2])
    
    # Get data excluding the first column (assuming it's an index)
    df_original_data = df_original.iloc[:, 1:]
    df_processed_data = df_processed.iloc[:, 1:]
    
    # Generate Color Map
    body_parts = df_original_data.columns.get_level_values(1).unique()
    body_part_colors = create_body_part_color_map(body_parts)
    
    # Get max values for frame size
    original_x_columns = df_original_data.xs('x', level=2, axis=1)
    original_y_columns = df_original_data.xs('y', level=2, axis=1)
    processed_x_columns = df_processed_data.xs('x', level=2, axis=1)
    processed_y_columns = df_processed_data.xs('y', level=2, axis=1)

    max_original_x = original_x_columns.max().max()
    max_original_y = original_y_columns.max().max()
    max_processed_x = processed_x_columns.max().max()
    max_processed_y = processed_y_columns.max().max()

    overall_max_x = max(max_original_x, max_processed_x)
    overall_max_y = max(max_original_y, max_processed_y)
    
    frame_width = int(overall_max_x + 100)
    frame_height = int(overall_max_y + 100)
    
    # Set up the plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(0, frame_width)
    ax.set_ylim(0, frame_height)

    def update(frame_idx):
        ax.clear()
        ax.set_xlim(0, frame_width)
        ax.set_ylim(0, frame_height)

        # Iterate through each body part and plot coordinates
        for body_part in body_parts:
            original_x = df_original_data[('DLC_snapshot-200000', body_part, 'x')].iloc[frame_idx]
            original_y = df_original_data[('DLC_snapshot-200000', body_part, 'y')].iloc[frame_idx]
            processed_x = df_processed_data[('DLC_snapshot-200000', body_part, 'x')].iloc[frame_idx]
            processed_y = df_processed_data[('DLC_snapshot-200000', body_part, 'y')].iloc[frame_idx]

            if pd.notna(original_x) and pd.notna(original_y):
                ax.plot(original_x, original_y, 'o', body_part_colors[body_part]['original'], markersize=5, zorder=2)

            if pd.notna(processed_x) and pd.notna(processed_y):
                ax.plot(processed_x, processed_y, 'o', color=body_part_colors[body_part]['processed'], markersize=10, zorder=1)
        
        # Save images if required
        if generate_images:
            image_filename = os.path.join(image_folder, f"frame_{frame_idx}.png")
            plt.savefig(image_filename, bbox_inches='tight', pad_inches=0.1)

        return ax,

    # Create animation (either video or gif based on selection)
    if animation_type in ['video', 'both']:
        video_filename = os.path.join(file_folder, f"{base_filename}_output_video.mp4")
        ani = FuncAnimation(fig, update, frames=len(df_original), interval=1000, blit=False)
        ani.save(video_filename, writer='ffmpeg', fps=fps)
        print(f"Video saved as {video_filename}")

    if animation_type in ['gif', 'both']:
        gif_filename = os.path.join(file_folder, f"{base_filename}_output_animation.gif")
        ani = FuncAnimation(fig, update, frames=len(df_original), interval=1000, blit=False)
        ani.save(gif_filename, writer='imagemagick', fps=fps)
        print(f"GIF saved as {gif_filename}")


# Access columns multindex

In [ ]:
import os
import numpy as np
import pandas as pd

input_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
processed_files = [f for f in os.listdir(processed_folder) if f.endswith('_processed.csv')]

for input_file in input_files:
    processed_file = f"{os.path.splitext(input_file)[0]}_processed.csv"
    if processed_file not in processed_files:
        continue
    
    df_original = pd.read_csv(os.path.join(input_folder, input_file), header=[0, 1, 2])
    df_processed = pd.read_csv(os.path.join(processed_folder, processed_file), header=[0, 1, 2])
    
    # Access the specific column
    #column_values = df_original[('DLC_snapshot-200000', 'tail', 'x')]

    # Print the values
    #print(column_values)  
    
    # Get all unique body parts from the second level of the header (skip the first column)
    df_original_data = df_original.iloc[:, 1:]  # Skipping the first column
    df_processed_data = df_processed.iloc[:, 1:]  # Skipping the first column
    
    # Extract all x and y columns
    original_x_columns = df_original_data.xs('x', level=2, axis=1)
    original_y_columns = df_original_data.xs('y', level=2, axis=1)
    processed_x_columns = df_processed_data.xs('x', level=2, axis=1)
    processed_y_columns = df_processed_data.xs('y', level=2, axis=1)

    # Find maximum values for original and processed data
    max_original_x = original_x_columns.max().max()
    max_original_y = original_y_columns.max().max()
    max_processed_x = processed_x_columns.max().max()
    max_processed_y = processed_y_columns.max().max()

    # Determine overall maxima
    overall_max_x = max(max_original_x, max_processed_x)
    overall_max_y = max(max_original_y, max_processed_y)

    # Print the results
    print(f"File: {input_file}")
    print(f"  Max X (Original): {max_original_x}, Max X (Processed): {max_processed_x}")
    print(f"  Max Y (Original): {max_original_y}, Max Y (Processed): {max_processed_y}")
    print(f"  Overall Max X: {overall_max_x}, Overall Max Y: {overall_max_y}")
    print("-" * 40)
    
    # Get all the bodyparts
    body_parts = df_original_data.columns.get_level_values(1).unique()

    # Iterate through each body part and get its x and y coordinates
    for body_part in body_parts:
        original_x_values = df_original_data[('DLC_snapshot-200000', body_part, 'x')]
        original_y_values = df_original_data[('DLC_snapshot-200000', body_part, 'y')]
        processed_x_values = df_processed_data[('DLC_snapshot-200000', body_part, 'x')]
        processed_y_values = df_processed_data[('DLC_snapshot-200000', body_part, 'y')]
        #print(f"Body Part: {body_part}")
        #print("ORIGINAL X Coordinates:", original_x_values.tolist())
        #print("ORIGINAL Y Coordinates:", original_y_values.tolist())
        #print("PROCESSED X Coordinates:", processed_x_values.tolist())
        #print("PROCESSED Y Coordinates:", processed_y_values.tolist())
        #print("-" * 40)
        
        
